# Processing - Aggregated Usage Data.ipynb

## Overview

This notebook loads **aggregated metering data** from `data/aggregated/` and produces
interactive charts for trend analysis, anomaly detection, and capacity planning.

**Aggregated data** means usage collapsed into equal time buckets (typically daily) across
all matching instances — a single value per `(metric, time-period)` pair. It is the right
data source when you need to:

- **Track trends** — how is a metric moving day over day or week over week?
- **Detect anomalies** — which days had unexpected spikes or drops?
- **Understand peak load** — what is the time-of-day usage pattern?
- **Correlate metrics** — does CPU usage track with memory consumption?

If you need to compare usage **across tenants, workspaces, or instances**,
use **`Processing - Grouped Usage Data.ipynb`** instead.
If you need individual record-level detail, use **`Processing - Raw Usage Data.ipynb`**.

---

### What this notebook produces

| # | Chart | What it answers |
|---|---|---|
| 5.1 | Synced multi-metric time-series dashboard | How are all metrics moving together over the window? |
| 5.2 | KPI summary cards | What is the current value, trend, and range for each metric? |
| 5.3 | Time-of-day usage heatmap | When during the day is usage highest (requires hourly data)? |
| 5.4 | Cumulative usage over time | What is the running total of consumption across the window? |
| 5.5 | Day-over-day change detection | Which days had the largest jumps or drops? |
| 5.6 | Resource distribution — histogram | How is usage distributed — is it uniform or skewed? |
| 5.7 | CPU vs Memory correlation scatter | Do high-CPU days also show high memory usage? |

---

### Prerequisites

- This notebook runs out of the box using the **included sample data** — no deployment or API access needed.
- To use your own data, run **`Fetch - Usage Data.ipynb`** first to populate `data/aggregated/<APP_DOMAIN>/<SERVICE_ID>/`, or point the configuration (Section 2) at an existing data directory.

All charts are fully interactive — hover for exact values, click legend items to toggle series,
drag to zoom, double-click to reset.


## 1. Imports

In [1]:
import json
import math
import os
import pathlib

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

## 2. Configuration

Load data path from environment variables or use defaults.
If you don't set `AGGREGATED_DATA_PATH` in `.env`, we'll use `data/aggregated/<APP_DOMAIN>/<SERVICE_ID>/`.
To use custom data, edit `.env` and set `AGGREGATED_DATA_PATH`.

No secrets are loaded or printed here.

In [2]:
if pathlib.Path(".env").exists():
    load_dotenv(".env")
else:
    load_dotenv(".env.template")

service_id   = os.getenv("SERVICE_ID", "").strip()
app_domain  = os.getenv("APP_DOMAIN", "").strip()

# Use nested path structure: data/aggregated/{APP_DOMAIN}/{SERVICE_ID}/
# APP_DOMAIN and SERVICE_ID are mandatory config values
if os.getenv("AGGREGATED_DATA_PATH"):
    data_path = os.getenv("AGGREGATED_DATA_PATH")  # Allow override via env var
else:
    data_path = f"data/aggregated/{app_domain}/{service_id}"

DATA_DIR    = pathlib.Path(data_path)

json_files = list(DATA_DIR.glob("*.json"))

if json_files:
    print(f"✓ Using data path: {data_path}")
    print(f"✓ Found {len(json_files)} JSON file(s)")
else:
    print(f"⚠ No JSON files found in: {DATA_DIR.resolve()}")
    print(f"\nTo use a different path, edit .env and set:")
    print(f"  AGGREGATED_DATA_PATH=/path/to/your/data")
    print(f"\nExample: AGGREGATED_DATA_PATH=sample_data/aggregated")

✓ Using data path: data/aggregated/apps.cluster.url.com/servicebrokercore
✓ Found 3 JSON file(s)


## 3. Load Aggregated Data

### Data format

Each JSON file has the shape returned by `GET /metering/services/{serviceId}/usage/aggregated`:

```json
{
  "params": {
    "metricId": "users",
    "transform": "sum",
    "groupByHrs": 24,
    "usageStart": 1782804739573,
    "usageEnd":   1785396739573,
    ...
  },
  "totalPeriodsCount": 3,
  "aggregatedMeteredUsagePeriods": [
    {
      "periodNumber":   1,
      "periodQuantity": 142.5,
      "periodStart":    1785110400000,
      "periodEnd":      1785196800000
    }
  ]
}
```

Each file is loaded as its own dataset. The `params` block is retained as metadata so charts
can label themselves with the correct metric, transform, and bucket size.

In [3]:
datasets = []
for path in sorted(DATA_DIR.glob("*.json")):
    with open(path) as f:
        data = json.load(f)
    periods = data.get("aggregatedMeteredUsagePeriods", [])
    if not periods:
        print(f"  {path.name} — no periods, skipping")
        continue
    params = data.get("params", {})
    df = pd.DataFrame(periods)
    df["periodStart"] = pd.to_datetime(df["periodStart"], unit="ms", utc=True)
    df["periodEnd"]   = pd.to_datetime(df["periodEnd"],   unit="ms", utc=True)
    df["periodQuantity"] = pd.to_numeric(df["periodQuantity"])
    datasets.append({
        "label":      path.stem,
        "metricId":   params.get("metricId", "unknown"),
        "transform":  params.get("transform", "unknown"),
        "groupByHrs": params.get("groupByHrs", 0),
        "df":         df,
    })
    print(
        f"  {path.name} — {len(df)} periods"
        f"  metric={params.get('metricId')}"
        f"  transform={params.get('transform')}"
        f"  groupByHrs={params.get('groupByHrs')}"
    )

  last_30d_daily_avg_instances.json — 16 periods  metric=instances  transform=avg  groupByHrs=24
  last_30d_daily_avg_users.json — 16 periods  metric=users  transform=avg  groupByHrs=24
  last_30d_daily_sum_api_calls.json — 18 periods  metric=api_calls  transform=sum  groupByHrs=24


## 4. Summary table

| Column | Meaning |
|---|---|
| **Metric** | Metric ID being measured |
| **Transform** | Aggregation applied by the API (`avg`, `max`, `sum`, etc.) |
| **Bucket (h)** | Width of each time bucket in hours |
| **Window** | Date range covered by the dataset |
| **Periods** | Buckets returned by the API |
| **Expected** | Buckets expected given the time window and `groupByHrs` |
| **Gaps** | Missing buckets (expected − reported) |
| **Total** | Sum of all `periodQuantity` values — overall consumption across the window |
| **Min / Avg / Max** | Per-period `periodQuantity` statistics |
| **Std Dev** | Standard deviation of `periodQuantity` — higher = more variable usage |
| **Trend** | Direction from first to last period — ↑ up  ↓ down  → flat |

In [4]:
rows = []
for ds in datasets:
    df = ds["df"].sort_values("periodStart")
    q  = df["periodQuantity"]

    window_hrs = (
        (df["periodEnd"].max() - df["periodStart"].min()).total_seconds() / 3600
    )
    bucket   = ds["groupByHrs"] or window_hrs
    expected = max(1, math.ceil(window_hrs / bucket))
    gaps     = max(0, expected - len(df))

    first, last = q.iloc[0], q.iloc[-1]
    delta = last - first
    if   delta >  0.01 * max(abs(first), abs(last), 1): trend = "↑"
    elif delta < -0.01 * max(abs(first), abs(last), 1): trend = "↓"
    else: trend = "→"

    t_start = df["periodStart"].min().strftime("%b %d")
    t_end   = df["periodEnd"].max().strftime("%b %d")

    rows.append({
        "Metric":     ds["metricId"],
        "Transform":  ds["transform"],
        "Bucket (h)": ds["groupByHrs"],
        "Window":     f"{t_start} – {t_end}",
        "Periods":    len(df),
        "Expected":   expected,
        "Gaps":       gaps,
        "Total":      round(q.sum(), 2),
        "Min":        round(q.min(), 2),
        "Avg":        round(q.mean(), 2),
        "Max":        round(q.max(), 2),
        "Std Dev":    round(q.std(), 2),
        "Trend":      trend,
    })

summary = pd.DataFrame(rows).sort_values(["Metric", "Bucket (h)"]).reset_index(drop=True)
summary

,Metric,Transform,Bucket (h),Window,Periods,Expected,Gaps,Total,Min,Avg,Max,Std Dev,Trend
0,api_calls,sum,24,Aug 11 – Aug 29,18,18,0,90751862,291215,5041770.11,5816066,1694023.07,↑
1,instances,avg,24,Aug 13 – Aug 29,16,16,0,16,1,1.00,1,0.00,→
2,users,avg,24,Aug 13 – Aug 29,16,16,0,16,1,1.00,1,0.00,→


## 5. Charts


### 5.1 Synced multi-metric time-series — line chart

### Computation

Builds one subplot row per loaded metric using `make_subplots` with `shared_xaxes=True`
so all time axes are synchronised. For each metric, three layers are drawn:
- **Raw series** — actual `periodQuantity` values per time bucket
- **Rolling average** — smoothed trend using a 7-period (or smaller) rolling window
- **Stability band** — shaded ±1 standard deviation region around the mean,
  showing how much the metric typically varies


In [5]:
n = len(datasets)
fig_51 = make_subplots(
    rows=n, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f"{ds['metricId']} ({ds['transform']} / {ds['groupByHrs']}h)"
        for ds in datasets
    ],
    vertical_spacing=0.06,
)
COLOURS = [
    "#636EFA", "#EF553B", "#00CC96",
    "#AB63FA", "#FFA15A", "#19D3F3",
]
for i, ds in enumerate(datasets, start=1):
    df     = ds["df"].sort_values("periodStart")
    colour = COLOURS[(i - 1) % len(COLOURS)]
    # Raw series
    fig_51.add_trace(
        go.Scatter(
            x=df["periodStart"],
            y=df["periodQuantity"],
            mode="lines+markers",
            name=f"{ds['metricId']} ({ds['transform']})",
            line={"color": colour, "width": 2},
            marker={"size": 5},
            hovertemplate="%{x|%b %d %H:%M}<br>%{y:.3f}<extra></extra>",
        ),
        row=i, col=1,
    )
    # ── D: 7-day rolling average (window = min(7, n_periods)) ────────────
    window = min(7, len(df))
    if window >= 2:
        roll = df["periodQuantity"].rolling(window=window, min_periods=1).mean()
        fig_51.add_trace(
            go.Scatter(
                x=df["periodStart"],
                y=roll,
                mode="lines",
                name=f"{ds['metricId']} {window}-period avg",
                line={"color": colour, "width": 2, "dash": "dash"},
                opacity=0.6,
                hovertemplate="%{x|%b %d %H:%M}<br>rolling avg: %{y:.3f}<extra></extra>",
                showlegend=True,
            ),
            row=i, col=1,
        )
    # ── Stability band: mean ± 1 std dev (shaded) ────────────────────────
    mean_val = df["periodQuantity"].mean()
    std_val  = df["periodQuantity"].std()
    if std_val > 0:
        x_vals = df["periodStart"].tolist()
        upper  = [mean_val + std_val] * len(x_vals)
        lower  = [mean_val - std_val] * len(x_vals)
        fig_51.add_trace(
            go.Scatter(
                x=x_vals + x_vals[::-1],
                y=upper + lower[::-1],
                fill="toself",
                fillcolor=f"rgba({int(colour[1:3],16)},{int(colour[3:5],16)},{int(colour[5:7],16)},0.10)",
                line={"color": "rgba(0,0,0,0)"},
                hoverinfo="skip",
                showlegend=False,
                name=f"{ds['metricId']} ±1 std",
            ),
            row=i, col=1,
        )
    fig_51.update_yaxes(title_text=f"Qty ({ds['transform']})", row=i, col=1)
fig_51.update_layout(
    height=300 * n,
    title_text="Aggregated usage over time — rolling avg + stability band (±1 std dev)",
    hovermode="x unified",
    showlegend=True,
    legend={"orientation": "h", "y": -0.12, "x": 0.5, "xanchor": "center"},
    margin={"b": 80},
);


### Chart Guide

**Purpose:** Shows how all loaded metrics are trending together over the query window,
with smoothing and a variability band to separate signal from noise.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Faceted line chart (one subplot per metric, shared X axis) |
| **X axis** | Period start timestamp (UTC) |
| **Y axis** | Usage quantity — one independent Y axis per metric |
| **Solid line** | Raw values per time bucket |
| **Dashed line** | Rolling average (smoothed trend) |
| **Shaded band** | ±1 standard deviation from the mean |

**Insights:** Points outside the shaded band are statistical outliers worth investigating.
If the rolling average is consistently rising, the metric is trending up over the window.


In [6]:
if fig_51 is not None:
    fig_51.show()


### 5.2 KPI summary cards — metric at a glance

### Computation

For each loaded metric, computes four summary statistics from `periodQuantity`:
- **Current** — the most recent period's value (`q.iloc[-1]`)
- **Average** — mean across all periods (`q.mean()`)
- **Peak** — highest single period value (`q.max()`)
- **Delta** — difference between current and average, shown with ▲▼ colouring

Each metric gets a two-row tile: a `go.Indicator` for the big number and a
sparkline `go.Scatter` underneath showing the trend over all periods.


In [7]:
n = len(datasets)
ncols = min(3, n)
nrows = (n + ncols - 1) // ncols
COLOURS = [
    "#636EFA", "#EF553B", "#00CC96",
    "#AB63FA", "#FFA15A", "#19D3F3",
]
def hex_to_rgba(hex_colour, alpha=0.15):
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)' — Plotly fillcolor compatible."""
    h = hex_colour.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"
# Each logical tile uses 2 subplot rows:
#   odd rows  → indicator (domain type required by go.Indicator)
#   even rows → sparkline (xy type)
specs = []
row_heights = []
for _ in range(nrows):
    specs.append([{"type": "domain"} for _ in range(ncols)])
    specs.append([{"type": "xy"}     for _ in range(ncols)])
    row_heights.extend([0.60, 0.40])
fig_52 = make_subplots(
    rows=nrows * 2,
    cols=ncols,
    specs=specs,
    vertical_spacing=0.04,
    horizontal_spacing=0.08,
    row_heights=row_heights,
)
for idx, ds in enumerate(datasets):
    col           = (idx % ncols) + 1
    tile_row      = (idx // ncols) * 2 + 1   # indicator row (1-based)
    sparkline_row = tile_row + 1              # sparkline row
    colour        = COLOURS[idx % len(COLOURS)]
    df      = ds["df"].sort_values("periodStart")
    q       = df["periodQuantity"]
    current = round(float(q.iloc[-1]), 2)
    avg_val = round(float(q.mean()), 2)
    peak    = round(float(q.max()), 2)
    unit    = ds["transform"]
    # ── Indicator (big number = most recent period value) ─────────────────
    fig_52.add_trace(
        go.Indicator(
            mode="number+delta",
            value=current,
            delta={
                "reference":   avg_val,
                "relative":    False,
                "valueformat": ".2f",
                "increasing":  {"color": "#00CC96"},
                "decreasing":  {"color": "#EF553B"},
            },
            title={
                "text": (
                    f"<b>{ds['metricId']}</b><br>"
                    f"<span style='font-size:0.75em;color:gray'>"
                    f"{unit} · {ds['groupByHrs']}h buckets</span><br>"
                    f"<span style='font-size:0.7em;color:gray'>"
                    f"avg {avg_val} &nbsp;|&nbsp; peak {peak}</span>"
                ),
                "align": "center",
            },
            number={"font": {"size": 40, "color": colour}, "valueformat": ".2f"},
        ),
        row=tile_row, col=col,
    )
    # ── Sparkline (filled area, clean/minimal) ────────────────────────────
    fig_52.add_trace(
        go.Scatter(
            x=df["periodStart"],
            y=q,
            mode="lines",
            line={"color": colour, "width": 2},
            fill="tozeroy",
            fillcolor=hex_to_rgba(colour),   # rgba fill at 15% opacity
            hovertemplate="%{x|%b %d %H:%M}<br>%{y:.2f}<extra></extra>",
            showlegend=False,
        ),
        row=sparkline_row, col=col,
    )
    # strip sparkline axes for a clean mini-chart look
    fig_52.update_xaxes(
        showticklabels=False, showgrid=False, zeroline=False,
        row=sparkline_row, col=col,
    )
    fig_52.update_yaxes(
        showticklabels=False, showgrid=False, zeroline=False,
        row=sparkline_row, col=col,
    )
fig_52.update_layout(
    height=300 * nrows,
    title_text="KPI summary — current value · avg · peak · trend",
    paper_bgcolor="white",
    plot_bgcolor="white",
    margin={"t": 60, "b": 20, "l": 20, "r": 20},
);


### Chart Guide

**Purpose:** Gives an at-a-glance summary of each metric's current state,
trend direction, and historical range without having to read a table.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | KPI indicator tiles with sparklines |
| **Big number** | Most recent period value |
| **Delta ▲▼** | Difference from the historical average (green = above, red = below) |
| **Sparkline** | All period values as a filled area — shape shows the trend |

**Insights:** A green delta means the latest period is above average — consumption is high.
A flat sparkline means usage is very stable. A right-skewed spike means a recent surge.


In [8]:
if fig_52 is not None:
    fig_52.show()


### 5.3 Time-of-day usage heatmap

### Computation

Requires hourly data (`groupByHrs=1`). For the first hourly metric loaded:
- Extracts `day-of-week` (0=Mon) and `hour-of-day` from each `periodStart` timestamp
- Groups by `(dow, hour)` and takes the **mean** `periodQuantity` across all weeks in the window
- Pivots into a 7 × 24 matrix (days × hours) and fills missing slots with zero
- Cells with no data are annotated with `—` to distinguish absence from zero


In [9]:
import numpy as np

DAYS  = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
HOURS = [f"{h:02d}:00" for h in range(24)]

hourly_datasets = [ds for ds in datasets if ds["groupByHrs"] == 1]

if not hourly_datasets:
    print("No hourly datasets found — time-of-day heatmap skipped.")
    print("Fetch a metric with groupByHrs=1 to see this chart.")
    fig_53 = None
else:
    # ── To show all hourly datasets, replace the next line with:
    # ── for ds in hourly_datasets:  (and indent the block below)
    ds = hourly_datasets[0]
    df = ds["df"].copy()
    df["dow"]  = df["periodStart"].dt.dayofweek
    df["hour"] = df["periodStart"].dt.hour
    pivot = (
        df.groupby(["dow", "hour"])["periodQuantity"]
        .mean().unstack(level="hour")
        .reindex(range(7)).reindex(columns=range(24)).fillna(0)
    )
    annot = pivot.round(1).astype(str)
    annot[pivot == 0] = "—"
    fig_53 = px.imshow(
        pivot.values,
        x=HOURS, y=[DAYS[i] for i in pivot.index],
        color_continuous_scale="Blues",
        title=f"{ds['metricId']} — avg {ds['transform']} by hour-of-day × day-of-week",
        labels={"x": "Hour of day (UTC)", "y": "Day of week",
                "color": f"{ds['metricId']} ({ds['transform']})"},
        aspect="auto",
    )
    for r, day in enumerate([DAYS[i] for i in pivot.index]):
        for c, hr in enumerate(HOURS):
            fig_53.add_annotation(
                x=hr, y=day, text=annot.iloc[r, c],
                showarrow=False, font={"size": 8, "color": "black"},
            )
    fig_53.update_xaxes(tickangle=-45, tickmode="linear")
    fig_53.update_layout(height=320, coloraxis_colorbar_title=ds["transform"]);


No hourly datasets found — time-of-day heatmap skipped.
Fetch a metric with groupByHrs=1 to see this chart.


### Chart Guide

**Purpose:** Reveals when during the week usage is highest — useful for capacity planning
and identifying whether load is evenly distributed or concentrated at specific times.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Annotated heatmap (7 rows × 24 columns) |
| **X axis** | Hour of day (UTC, 00:00 → 23:00) |
| **Y axis** | Day of week (Mon → Sun) |
| **Colour** | Mean usage quantity — darker = higher average usage |
| **Cell text** | Rounded mean value; `—` = no data for that slot |

**Insights:** Dark columns at specific hours indicate peak load times.
A uniformly dark row on a weekday with light weekends suggests business-hours usage patterns.
> Requires hourly data (`groupByHrs=1`). If skipped, re-fetch with a 1h bucket size.


In [10]:
if fig_53 is not None:
    fig_53.show()


### 5.4 Cumulative usage over time

### Computation

Filters to datasets with `transform=sum` (cumulative semantics only make sense for totals).
For the first matching dataset, sorts periods chronologically and applies `cumsum()`
to `periodQuantity` to produce a running total over the window.


In [11]:
sum_datasets = [ds for ds in datasets if ds["transform"] == "sum"]

if not sum_datasets:
    print("No sum-transformed datasets — cumulative chart skipped.")
    fig_54 = None
else:
    # ── To show all sum datasets, replace the next line with:
    # ── for ds in sum_datasets:  (and indent the block below)
    ds  = sum_datasets[0]
    df  = ds["df"].sort_values("periodStart")
    fig_54 = go.Figure()
    fig_54.add_trace(go.Scatter(
        x=df["periodStart"],
        y=df["periodQuantity"].cumsum(),
        mode="lines+markers",
        name=f"{ds['metricId']} · {ds['transform']} · {ds['groupByHrs']}h",
        hovertemplate="%{x|%b %d %H:%M}<br>cumulative: %{y:.3f}<extra></extra>",
    ))
    fig_54.update_layout(
        title="Cumulative usage over time (sum datasets only)",
        xaxis_title="Period start (UTC)",
        yaxis_title="Cumulative quantity",
        hovermode="x unified",
        plot_bgcolor="white",
        margin=dict(t=80, b=60, l=60, r=40),
    );


### Chart Guide

**Purpose:** Shows the total accumulated consumption from the start of the window to each point
in time — the running bill or running usage counter.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Line chart (monotonically increasing for sum metrics) |
| **X axis** | Period start timestamp |
| **Y axis** | Running total (cumulative sum of period quantities) |

**Insights:** A steep slope means rapid consumption. A flat segment means no usage was recorded
in that interval — either genuinely zero activity or a reporting gap.
> Only available for `transform=sum` metrics. Other transforms are skipped.


In [12]:
if fig_54 is not None:
    fig_54.show()


### 5.5 Day-over-day change detection — bar chart

### Computation

For the first loaded dataset, sorts periods chronologically and computes `diff()` on
`periodQuantity` to get the change from one period to the next.
Positive deltas are coloured green, negative red. The zero line is drawn explicitly.


In [13]:
COLOURS_POS = "#00CC96"
COLOURS_NEG = "#EF553B"

# ── To show all datasets, replace the next line with:
# ── for ds in datasets:  (and indent the block below)
ds  = datasets[0]
df  = ds["df"].sort_values("periodStart").copy()
fig_55 = None
if len(df) < 2:
    print(f"{ds['metricId']}: not enough periods for change detection — skipped.")
else:
    df["delta"]  = df["periodQuantity"].diff()
    df           = df.dropna(subset=["delta"])
    df["label"]  = df["periodStart"].dt.strftime("%b %d")
    sorted_labels = df.sort_values("periodStart")["label"].tolist()
    colours = [COLOURS_POS if v >= 0 else COLOURS_NEG for v in df["delta"]]
    fig_55 = go.Figure(go.Bar(
        x=df["label"], y=df["delta"],
        marker_color=colours,
        text=[f"{v:+.2f}" for v in df["delta"]],
        textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>Change: %{y:+.3f}<extra></extra>",
    ))
    fig_55.add_hline(y=0, line_width=1, line_color="#888")
    fig_55.update_xaxes(categoryorder="array", categoryarray=sorted_labels, tickangle=-45)
    fig_55.update_layout(
        title=f"{ds['metricId']} — period-over-period change ({ds['transform']} / {ds['groupByHrs']}h)",
        yaxis_title=f"Δ {ds['transform']}", xaxis_title="Period start",
        height=420, plot_bgcolor="white", paper_bgcolor="white",
        margin={"t": 60, "b": 80, "l": 60, "r": 60},
    );


### Chart Guide

**Purpose:** Highlights which periods had the largest jumps or drops, making it easy
to pinpoint anomalous days without scanning a raw table.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Diverging bar chart (positive/negative from zero) |
| **X axis** | Date |
| **Y axis** | Change in quantity vs the previous period |
| **Green bars** | Increase from previous period |
| **Red bars** | Decrease from previous period |

**Insights:** Alternating green/red bars indicate noisy or volatile data.
A single tall red bar after steady greens is a sharp drop worth investigating.


In [14]:
if fig_55 is not None:
    fig_55.show()


### 5.6 Resource distribution — histogram

### Computation

For the first loaded dataset, bins all `periodQuantity` values into a histogram.
The number of bins is capped between 5 and 30 based on unique value count.
Three vertical reference lines are overlaid: the **mean** (red solid) and
**±1 standard deviation** (orange dashed), showing how spread the distribution is.
A rug plot along the X axis marks individual data points.


In [15]:
# ── To show all datasets, replace the next line with:
# ── for ds in datasets:  (and indent the block below)
ds  = datasets[0]
df  = ds["df"].copy()
q   = df["periodQuantity"]
nbins = max(5, min(30, q.nunique()))
fig_56 = px.histogram(
    df, x="periodQuantity", nbins=nbins, marginal="rug",
    title=f"{ds['metricId']} — utilisation distribution ({ds['transform']} / {ds['groupByHrs']}h, {len(df)} periods)",
    labels={"periodQuantity": f"{ds['metricId']} ({ds['transform']})"},
    color_discrete_sequence=["#636EFA"],
)
mean_v, std_v = q.mean(), q.std()
for xval, label, colour, dash in [
    (mean_v,         f"mean ({mean_v:.2f})",           "#EF553B", "solid"),
    (mean_v + std_v, f"+1σ ({mean_v+std_v:.2f})",     "#FFA15A", "dash"),
    (mean_v - std_v, f"−1σ ({max(0,mean_v-std_v):.2f})", "#FFA15A", "dash"),
]:
    if xval >= 0:
        fig_56.add_vline(x=xval, line_dash=dash, line_color=colour,
            annotation_text=label, annotation_position="top right",
            annotation_font_size=10)
fig_56.update_layout(height=380, bargap=0.05, plot_bgcolor="white", paper_bgcolor="white",
    margin=dict(t=80, b=60, l=60, r=40));


### Chart Guide

**Purpose:** Shows how usage values are distributed across all periods —
whether consumption is concentrated at a typical level or highly variable.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Histogram with rug plot |
| **X axis** | Usage quantity value |
| **Y axis** | Count of periods in that bin |
| **Red line** | Mean across all periods |
| **Orange dashed lines** | ±1 standard deviation from the mean |
| **Rug marks** | Individual period values along the X axis |

**Insights:** A narrow peak near the mean means stable, predictable usage.
A wide spread or bimodal shape means usage varies significantly — worth understanding why.


In [16]:
if fig_56 is not None:
    fig_56.show()


### 5.7 CPU vs Memory correlation — scatter chart

### Computation

Requires both `cpu_cores` and `memory_gb` datasets to be loaded.
Since memory is typically fetched at hourly resolution and CPU at daily,
both are aggregated to a **daily average** before joining on date.
A linear regression trend line is fitted with `numpy.polyfit` and overlaid.
If `memory_gb` is constant (no variance), an explanatory annotation is added.


In [17]:
# Align cpu_cores and memory_gb by period start (nearest-period join)
cpu_ds  = next((d for d in datasets if d["metricId"] == "cpu_cores"),  None)
mem_ds  = next((d for d in datasets if d["metricId"] == "memory_gb"),  None)

if cpu_ds is None or mem_ds is None:
    missing = [m for m, d in [("cpu_cores", cpu_ds), ("memory_gb", mem_ds)] if d is None]
    print(f"CPU vs Memory scatter skipped — missing dataset(s): {missing}")
    fig_57 = None
    print("Load both cpu_cores and memory_gb files to enable this chart.")
else:
    df_cpu = cpu_ds["df"][["periodStart", "periodQuantity"]].rename(
        columns={"periodQuantity": "cpu_cores"}
    ).copy()
    df_mem = mem_ds["df"][["periodStart", "periodQuantity"]].rename(
        columns={"periodQuantity": "memory_gb"}
    ).copy()

    # Round periodStart to the daily bucket for the merge (handles hourly mem data)
    df_cpu["day"] = df_cpu["periodStart"].dt.normalize()
    df_mem["day"] = df_mem["periodStart"].dt.normalize()

    # memory_gb is hourly (groupByHrs:1), cpu_cores is daily (groupByHrs:24)
    # Aggregate both to daily average for correlation
    df_cpu_d = df_cpu.groupby("day", as_index=False)["cpu_cores"].mean()
    df_mem_d = df_mem.groupby("day", as_index=False)["memory_gb"].mean()

    merged = df_cpu_d.merge(df_mem_d, on="day", how="inner")
    merged["date_label"] = merged["day"].dt.strftime("%b %d")

    if merged.empty:
        print("No overlapping periods between cpu_cores and memory_gb — scatter skipped.")
    else:
        fig_57 = px.scatter(
            merged,
            x="cpu_cores",
            y="memory_gb",
            color="date_label",
            text="date_label",
            title="CPU vs Memory correlation — one point per day (daily average)",
            labels={
                "cpu_cores":  "CPU cores (avg)",
                "memory_gb":  "Memory GB (avg)",
                "date_label": "Date",
            },
            color_discrete_sequence=px.colors.qualitative.Plotly,
        )
        fig_57.update_traces(textposition="top center", marker={"size": 10})
        
        # Check if memory_gb is constant
        if merged["memory_gb"].nunique() == 1:
            fig.add_annotation(
                text=f"<i>Memory GB is constant ({merged['memory_gb'].iloc[0]} GB) — no variance in data</i>",
                xref="paper", yref="paper",
                x=0.5, y=-0.22,
                showarrow=False,
                font={"size": 11, "color": "#999"},
                xanchor="center",
            )
            # Add bottom margin to accommodate annotation
            fig_57.update_layout(margin=dict(l=60, r=60, t=80, b=100))

        # Add a linear-regression trend line
        if len(merged) >= 2:
            import numpy as np
            m, b = np.polyfit(merged["cpu_cores"], merged["memory_gb"], 1)
            x_line = [merged["cpu_cores"].min(), merged["cpu_cores"].max()]
            y_line = [m * x + b for x in x_line]
            fig_57.add_trace(go.Scatter(
                x=x_line, y=y_line,
                mode="lines",
                line={"dash": "dot", "color": "#888", "width": 1.5},
                name=f"trend (slope={m:.2f})",
                hoverinfo="skip",
            ))

        fig_57.update_layout(
            height=480,
            plot_bgcolor="white",
            paper_bgcolor="white",
            margin=dict(t=80, b=80, l=60, r=40),
        );
        

CPU vs Memory scatter skipped — missing dataset(s): ['cpu_cores', 'memory_gb']
Load both cpu_cores and memory_gb files to enable this chart.


### Chart Guide

**Purpose:** Tests whether CPU and memory usage move together —
a strong correlation suggests they are driven by the same workload.

**Chart Attributes**

| | |
|---|---|
| **Chart type** | Scatter plot with trend line |
| **X axis** | CPU cores (daily average) |
| **Y axis** | Memory GB (daily average) |
| **Colour** | Date — each point is one day |
| **Dotted line** | Linear regression trend |

**Insights:** Points clustered tightly around the trend line indicate strong correlation.
Points far from the line are days where CPU and memory diverged — worth investigating.
> Skipped automatically if either `cpu_cores` or `memory_gb` is not loaded.


In [18]:
if fig_57 is not None:
    fig_57.show()
